# MGMT298D: Science and Strategy of AI
### Week 5B - Time Series Forecasting
### Application: Retail Demand Prediction

## Import Libraries and Data

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

!pip install prophet -q
from prophet import Prophet

# Load retail sales data
url = 'https://raw.githubusercontent.com/facebook/prophet/main/examples/example_retail_sales.csv'
df = pd.read_csv(url)
df['ds'] = pd.to_datetime(df['ds'])

print(f"Dataset: {len(df)} days of sales data")
print(df.head())

## Visualize Sales Over Time

In [ ]:
plt.figure(figsize=(16, 6))
plt.plot(df['ds'], df['y'])
plt.title('Daily Retail Sales Over Time')
plt.xlabel('Date')
plt.ylabel('Sales')
plt.grid(True)
plt.show()

print("Key patterns: upward trend, yearly seasonality (Christmas spikes)")

## Build Forecasting Model with Prophet

In [ ]:
# Prophet expects columns: 'ds' (date) and 'y' (value)
model = Prophet(yearly_seasonality=True, weekly_seasonality=True, daily_seasonality=False)
model.fit(df)

# Forecast next 365 days
future = model.make_future_dataframe(periods=365)
forecast = model.predict(future)

print("Forecast preview:")
print(forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].tail())

## Visualize Forecast

In [ ]:
fig1 = model.plot(forecast, figsize=(16, 6))
plt.title('Sales Forecast for Next Year')
plt.xlabel('Date')
plt.ylabel('Sales')
plt.show()

## Decompose Forecast Components

In [ ]:
fig2 = model.plot_components(forecast, figsize=(16, 8))
plt.show()

print("Component insights:")
print("  - Trend: steady upward growth")
print("  - Weekly: peaks on weekends, lowest on Tuesdays")
print("  - Yearly: Christmas spike, smaller spring bump")

## Evaluate Model Performance

In [ ]:
from prophet.diagnostics import cross_validation, performance_metrics

# Cross-validation: test on 180-day horizons
df_cv = cross_validation(model, initial='730 days', period='30 days', horizon='180 days')
df_p = performance_metrics(df_cv)

mape = df_p['mape'].mean() * 100
print(f"Mean Absolute Percentage Error (MAPE): {mape:.2f}%")
print(f"Interpretation: forecasts are within ~{mape:.0f}% of actual sales on average")